# Занятие 3. Аналитика: витрины, DAG и Metabase

Путь данных в этом занятии:

```
iceberg.retail.sales_silver ──Spark SQL──▶ витрина в Postgres (dwh.public.mart_*) ──▶ график в Metabase
```

План:

1. **Пример** — витрина «выручка по дням и регионам» целиком, от запроса до таблицы в Postgres.
2. **Задачи 1–3** — три витрины самостоятельно.
3. **Задача 4** — одну из своих витрин обернуть в DAG Airflow по образцу.
4. **Задача 5** — графики в Metabase.

**Перед началом:** таблица `iceberg.retail.sales_silver` заполнена (занятие 2),
DAG `retail_sales_bronze` загрузил все часы, silver обновлён после этого.

In [ ]:
import sys
sys.path.append("../host")
from spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark("lab-03")


DWH = dict(url="jdbc:postgresql://postgres:5433/dwh", user="course", password="course_pass",
           driver="org.postgresql.Driver")


def write_to_dwh(df, table):
    """Перезаписывает таблицу public.<table> в базе dwh (её потом читает Metabase)."""
    (df.write.format("jdbc").mode("overwrite")
       .option("truncate", "true")        # очистить таблицу, но не удалять её
       .options(dbtable=f"public.{table}", **DWH)
       .save())
    print(f"dwh.public.{table}: {df.count()} строк")


def read_from_dwh(table):
    return spark.read.format("jdbc").options(dbtable=f"public.{table}", **DWH).load()

## Метрики

| метрика | как считать |
|---|---|
| **Выручка** (чистая) | сумма `price_paid_kop` продаж без отменённых (`SALE` и `NOT is_cancelled`) минус сумма возвратов (`RETURN`) |
| **Чек** | уникальный `receipt_id` среди неотменённых продаж |
| **Средний чек** | выручка продаж / число чеков. Возвраты не участвуют: у возврата свой чек |

Суммы хранятся в **копейках** (целые числа, без ошибок округления). В рубли переводим
только в витрине: `CAST(... / 100 AS DECIMAL(18, 2))` — ровно 2 знака после запятой,
без экспоненты вида `5.92E8`, как дал бы `ROUND`.

Таблицы: `iceberg.retail.sales_silver` (операции), `iceberg.retail.stores` (магазины:
город, регион, формат), `iceberg.retail.categories`, `iceberg.retail.products`.

In [ ]:
spark.table("iceberg.retail.sales_silver").limit(5).toPandas()

## Пример. Выручка по дням и регионам

Шаг 1 — запрос. Выручка складывается из трёх случаев, поэтому внутри `SUM` стоит `CASE`.
Чеки считаем через `COUNT(DISTINCT ...)`: в одном чеке много позиций.

In [ ]:
daily_revenue = spark.sql("""
    SELECT
        s.event_date,
        st.region,
        CAST(SUM(CASE
                    WHEN s.operation_type = 'SALE' AND NOT s.is_cancelled THEN s.price_paid_kop
                    WHEN s.operation_type = 'RETURN' THEN -s.price_paid_kop
                    ELSE 0
                  END) / 100 AS DECIMAL(18, 2)) AS revenue_rub,
        COUNT(DISTINCT CASE WHEN s.operation_type = 'SALE' AND NOT s.is_cancelled
                            THEN s.receipt_id END) AS receipts
    FROM iceberg.retail.sales_silver AS s
    JOIN iceberg.retail.stores AS st ON st.store_id = s.store_id
    GROUP BY s.event_date, st.region
""")

daily_revenue.orderBy("event_date", F.desc("revenue_rub")).show(50, truncate=False)

Шаг 2 — запись в Postgres. Функция `write_to_dwh` (ячейка в начале ноутбука) каждый раз
перезаписывает таблицу целиком. Витрина маленькая, так проще всего: повторный запуск
даёт тот же результат.

In [ ]:
write_to_dwh(daily_revenue, "mart_daily_revenue")
read_from_dwh("mart_daily_revenue").orderBy("event_date", "region").show(5)

Шаг 3 — регулярный пересчёт. Тот же запрос оформлен как скрипт
[jobs/retail/mart_daily_revenue.py](../jobs/retail/mart_daily_revenue.py), а DAG
[dags/retail_mart_daily_revenue.py](../dags/retail_mart_daily_revenue.py) запускает его на
кластере через `spark-submit`. Откройте оба файла — отличий от ячеек выше почти нет:

* вместо `get_spark(...)` — `SparkSession.builder.getOrCreate()`: настройки кластера
  приходят от `spark-submit`;
* в конце — `spark.stop()`.

В Airflow (http://localhost:8080) включите DAG `retail_mart_daily_revenue` и нажмите
**Trigger DAG**. Перед этим остановите сессию ноутбука (`spark.stop()`) или дождитесь,
пока DAG получит ядра.

## Задача 1. Средний чек по регионам и форматам магазинов

Постройте витрину `mart_avg_check` с колонками:

| колонка | смысл |
|---|---|
| `region` | регион |
| `format` | формат магазина: гипермаркет / супермаркет / у дома |
| `is_loyalty` | участник бонусной программы |
| `receipts` | число чеков |
| `avg_check_rub` | средний чек, руб. (2 знака) |

Вопросы: где средний чек выше всего? Во сколько раз чек участника программы больше?

*Подсказка:* сначала соберите чеки — `GROUP BY receipt_id` с суммой позиций (только
неотменённые продажи). Затем усредните суммы чеков по группам. Регион и формат возьмите
из `iceberg.retail.stores`.

In [ ]:
# Задача 1
avg_check = spark.sql("""
    -- ваш запрос
""")

avg_check.show(50, truncate=False)
# write_to_dwh(avg_check, "mart_avg_check")

## Задача 2. Топ-5 товаров в каждой категории

Постройте витрину `mart_top_products`: для каждой категории пять товаров с наибольшей
чистой выручкой за весь период.

| колонка | смысл |
|---|---|
| `group_name`, `category_name` | группа и категория |
| `rank` | место в категории: 1..5 |
| `product_name` | название товара |
| `quantity` | продано штук (без отменённых) |
| `revenue_rub` | чистая выручка, руб. |

*Подсказка:* посчитайте выручку по каждому товару, затем пронумеруйте товары внутри
категории оконной функцией
`ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY revenue DESC)` и оставьте `rank <= 5`.

In [ ]:
# Задача 2
top_products = spark.sql("""
    -- ваш запрос
""")

top_products.show(50, truncate=False)
# write_to_dwh(top_products, "mart_top_products")

## Задача 3. Способы оплаты по регионам

Постройте витрину `mart_payment_mix`: как в каждом регионе распределяется выручка продаж
по способам оплаты (`CARD`, `SBP`, `CASH`, `BONUS`).

| колонка | смысл |
|---|---|
| `region` | регион |
| `payment_method` | способ оплаты |
| `revenue_rub` | выручка неотменённых продаж, руб. |
| `share_pct` | доля способа оплаты в выручке региона, % (сумма по региону = 100) |

Вопрос: где наличные ещё популярны, а где их почти не осталось?

*Подсказка:* долю удобно посчитать оконной суммой
`revenue / SUM(revenue) OVER (PARTITION BY region)`.

In [ ]:
# Задача 3
payment_mix = spark.sql("""
    -- ваш запрос
""")

payment_mix.show(80, truncate=False)
# write_to_dwh(payment_mix, "mart_payment_mix")

In [ ]:
spark.stop()   # освобождаем кластер для DAG

## Задача 4. Своя витрина в DAG

Оберните одну из своих витрин (например, из задачи 1) в DAG по образцу примера.

1. Скопируйте `jobs/retail/mart_daily_revenue.py` в `jobs/retail/mart_avg_check.py`.
   Замените SQL-запрос на свой, в `dbtable` укажите `public.mart_avg_check`.
2. Скопируйте `dags/retail_mart_daily_revenue.py` в `dags/retail_mart_avg_check.py`.
   Поменяйте три строки: `dag_id="retail_mart_avg_check"`,
   `application="/opt/jobs/retail/mart_avg_check.py"` и `name`.
3. Через ~30 секунд DAG появится в http://localhost:8080. Включите его и нажмите
   **Trigger DAG**.
4. Если задача упала: откройте её и вкладку **Logs** — ошибка Spark будет в конце лога.
   Частая причина — опечатка в имени таблицы или колонки.

Проверить, что витрина обновилась, можно запросом в Metabase или ячейкой ниже (заново
создаёт сессию Spark).

In [ ]:
spark = get_spark("lab-03-check")
read_from_dwh("mart_avg_check").show(5)
spark.stop()

## Задача 5. Графики в Metabase

Откройте http://localhost:3000. Если база `dwh` ещё не подключена — см. README стенда,
раздел «Metabase».

**Новые таблицы Metabase видит не сразу:** схему базы он перечитывает раз в час. Чтобы
витрины появились сейчас: ⚙ **Admin settings → Databases → dwh → Sync database schema**,
подождите полминуты и обновите страницу.

**График 1 — выручка по дням и регионам:**

1. **New → Question** → база `dwh` → таблица `Mart Daily Revenue`.
2. **Summarize:** `Sum of Revenue Rub`, группировка по `Event Date: Day` и `Region`.
3. **Visualization → Line**. Сохраните вопрос: *Выручка по дням и регионам*.

**График 2 — по своей витрине**, например средний чек:

1. `Mart Avg Check` → **Summarize:** `Average of Avg Check Rub` по `Region` и `Is Loyalty`.
2. **Visualization → Bar**. Сохраните.

(Среднее средних чеков — не то же самое, что средний чек региона. Для графика сойдёт,
но подумайте, как посчитать точно: для этого в витрине есть `receipts`.)

**Дашборд:** **New → Dashboard** → *Розничная сеть* → добавьте оба вопроса.

Когда DAG витрины отработает ещё раз, дашборд покажет новые цифры сам: Metabase читает
таблицы в Postgres при каждом открытии.